# 07c -- Ablación de features extendidas (fase B, parte 2)

¿Las features de `07b` mejoran el baseline? **Hiperparámetros y pesos de ensamble fijos** (los de
`MODELOS_POR_RAMA`, sin búsqueda), mismo protocolo walk-forward que 08 (entrena con folds anteriores,
prueba en el fold actual), mismas filas.

Diseño para no repetir el problema de selección sobre la evaluación:
1. **Replicación**: con las features v1 este notebook debe reproducir las predicciones de 08. Si no, se detiene.
2. **6 configuraciones**: `v1`, `v1` + cada grupo de features (memoria, momentum, contexto, calendario) y `v2_completa`.
3. **Comparación pareada** contra `v1` (mejora relativa con IC95 por bootstrap de productos), no solo contra el piso.
4. **Regla mecánica** (pre-registrada): la configuración se elige con los folds de búsqueda (2-4) y se
   confirma **una vez** en el fold 5 (`evaluar_nivel1.POLITICA_HOLDOUT`). Se adopta solo si el IC95 de la
   mejora sobre v1 en 2-4 queda sobre 0 **y** el fold 5 no la contradice.

Cadena: ... -> `07b_features_extendidas` -> **`07c_ablacion_features`** -> 10-13

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import DW, registrar_huella, verificar_huella, purgar_entrenamiento
import evaluar_nivel1 as ev
from exportar_modelos_nivel1 import (MODELOS_POR_RAMA, RAMA_A_CONDICION, FEATURES as F1,
                                     _entrena_lightgbm_q, _entrena_tweedie_completo, _alpha_negocio_de)

import json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', category=Warning, module='sklearn')   # ConvergenceWarning del GLM, ya conocido en 08

pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

EXPERIMENTO = 'EDA-fix-nivel1-07c-ablacion-features'
Y, PISO = 'target_demanda_15d', 'pred_media_movil_15d'
LLAVES = ['codigo_item', 'sucursal', 'fecha_origen']
B, SEMILLA = 2000, 0

verificar_huella('huella_07b.json', ['matriz_as_of_v2.parquet'])
m = pd.read_parquet(f'{DW}/matriz_as_of_v2.parquet')
GRUPOS = {
    'memoria': ['trail_7', 'trail_28', 'ewma_14', 'ewma_28', 'dias_con_venta_15'],
    'momentum': ['ratio_trail_nivel'],
    'contexto': ['cat_ratio', 'share_cat', 'suc_ratio'],
    'calendario': ['n_festivos_ventana', 'n_inicio_mes_ventana'],
}
CONFIGS = {'v1': list(F1)}
for g, cols in GRUPOS.items():
    CONFIGS[f'v1+{g}'] = list(F1) + cols
CONFIGS['v2_completa'] = list(F1) + [c for cols in GRUPOS.values() for c in cols]
assert all(c in m.columns for cols in CONFIGS.values() for c in cols)
print({k: len(v) for k, v in CONFIGS.items()})
print('filas', m.shape, '| folds', sorted(m['fold_id'].unique()))

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 18:35:55 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Huella verificada OK contra huella_07b.json.
{'v1': 14, 'v1+memoria': 19, 'v1+momentum': 15, 'v1+contexto': 17, 'v1+calendario': 16, 'v2_completa': 25}
filas (262429, 38) | folds [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 1. Entrenamiento walk-forward por configuración

Mismo esquema que 08: para el fold `k`, entrena con todas las filas de folds `< k` y predice el fold `k`.
Ramas: `intermitente` = LightGBM cuantílico q50; ramas suaves = ensamble `w·Tweedie + (1-w)·LightGBM q50`.

In [2]:
def predecir_q50(rama, cols, train, test):
    spec = MODELOS_POR_RAMA[rama]
    X, y, Xt = train[cols], train[Y], test[cols]
    if spec['tipo'] == 'lightgbm':
        return _entrena_lightgbm_q(X, y, 0.5, spec['hparams']).predict(Xt)
    tw, _ = _entrena_tweedie_completo(X, y, spec['hparams_tweedie'], _alpha_negocio_de(rama))
    lg = _entrena_lightgbm_q(X, y, 0.5, spec['hparams_lgb'])
    w = spec['peso_tweedie']
    return w * np.clip(tw.predict(Xt), 0, None) + (1 - w) * lg.predict(Xt)

FOLDS_EVAL = [2, 3, 4, 5]
partes, t0 = [], time.time()
for rama, cond in RAMA_A_CONDICION.items():
    dr = m.loc[cond(m)]
    for f in FOLDS_EVAL:
        tr, te = purgar_entrenamiento(dr, f, verbose=False), dr[dr['fold_id'] == f]   # sin ventanas solapadas con el test
        if len(tr) < ev.MIN_FILAS_ENTRENAMIENTO:
            print(f'{rama:22s} fold {f}: train={len(tr)} < {ev.MIN_FILAS_ENTRENAMIENTO} tras el purge -- omitida (igual que 08)'); continue
        blk = te[LLAVES + ['fold_id', Y, PISO]].copy()
        blk['rama'] = rama
        for nombre, cols in CONFIGS.items():
            blk[f'pred_{nombre}'] = predecir_q50(rama, cols, tr, te)
        partes.append(blk)
        print(f'{rama:22s} fold {f}: train={len(tr):>7,} test={len(te):>6,}  ({time.time()-t0:,.0f}s)')
res = pd.concat(partes, ignore_index=True)
print(res.shape)

suave_no_perecedero    fold 2: train=    260 test=   871  (6s)


suave_no_perecedero    fold 3: train=  1,129 test=   315  (13s)


suave_no_perecedero    fold 4: train=  1,448 test=   600  (21s)


suave_no_perecedero    fold 5: train=  2,051 test=   624  (29s)
suave_perecedero       fold 2: train=168 < 200 tras el purge -- omitida (igual que 08)


suave_perecedero       fold 3: train=    714 test=   200  (33s)


...

suave_perecedero       fold 5: train=  1,316 test=   444  (41s)


intermitente           fold 2: train= 20,152 test=68,198  (58s)


intermitente           fold 3: train= 88,142 test=28,850  (84s)


intermitente           fold 4: train=116,468 test=59,600  (112s)


intermitente           fold 5: train=175,878 test=76,056  (145s)
(236158, 13)


## 2. Control de replicación contra 08

Con las features v1, este notebook debe dar (casi) las mismas predicciones que `evaluacion_nivel_1.parquet`.
Sin esto, cualquier diferencia entre v1 y v2 podría ser del procedimiento y no de las features.

In [3]:
d08 = pd.read_parquet(f'{DW}/evaluacion_nivel_1.parquet')[LLAVES + ['rama', 'pred_q50']]
chk = res.merge(d08, on=LLAVES + ['rama'], how='inner', validate='one_to_one')
assert len(chk) == len(res), f'no coinciden las filas: {len(chk)} vs {len(res)}'
filas = []
for rama, g in chk.groupby('rama'):
    filas.append(dict(rama=rama, wape_08=ev.wape(g[Y], g['pred_q50']), wape_07c_v1=ev.wape(g[Y], g['pred_v1']),
                      dif_abs_max=float((g['pred_q50'] - g['pred_v1']).abs().max())))
rep = pd.DataFrame(filas).set_index('rama')
print(rep)
assert (rep['wape_08'] - rep['wape_07c_v1']).abs().max() < 0.003, 'NO reproduce 08 -- revisar antes de comparar'
print('replicación OK (diferencia de WAPE < 0.003 en las 3 ramas)')

                     wape_08  wape_07c_v1  dif_abs_max
rama                                                  
intermitente           0.433        0.433        0.000
suave_no_perecedero    0.188        0.188        0.000
suave_perecedero       0.141        0.141        0.000
replicación OK (diferencia de WAPE < 0.003 en las 3 ramas)


## 3. Resultados: cada configuración contra el piso y contra v1

`mejora_vs_piso` = frente a la media móvil. `mejora_vs_v1` = frente al mismo modelo con las 14 features
originales (positivo = las features nuevas ayudan). Ambas con IC95 (bootstrap de productos).

In [4]:
filas = []
for rama, g in res.groupby('rama'):
    for cfg in CONFIGS:
        col = f'pred_{cfg}'
        p = ev.mejora_con_ic(g, Y, col, PISO, B, SEMILLA)
        fila = dict(rama=rama, config=cfg, wape=p['wape_modelo'], wape_piso=p['wape_piso'],
                    mejora_vs_piso=p['mejora_rel'], piso_ic_bajo=p['ic_bajo'], piso_ic_alto=p['ic_alto'])
        if cfg != 'v1':
            q = ev.mejora_con_ic(g, Y, col, 'pred_v1', B, SEMILLA)
            b24 = ev.mejora_con_ic(g[g['fold_id'].isin(ev.FOLDS_BUSQUEDA)], Y, col, 'pred_v1', B, SEMILLA)
            h5 = ev.mejora_con_ic(g[g['fold_id'] == ev.FOLD_HOLDOUT], Y, col, 'pred_v1', B, SEMILLA)
            fila.update(mejora_vs_v1=q['mejora_rel'], v1_ic_bajo=q['ic_bajo'], v1_ic_alto=q['ic_alto'],
                        mejora_vs_v1_f2a4=b24['mejora_rel'], f2a4_ic_bajo=b24['ic_bajo'], f2a4_ic_alto=b24['ic_alto'],
                        mejora_vs_v1_f5=h5['mejora_rel'], f5_ic_bajo=h5['ic_bajo'], f5_ic_alto=h5['ic_alto'])
        filas.append(fila)
tabla = pd.DataFrame(filas)
for rama in tabla['rama'].unique():
    print(f'\n=== {rama} ===')
    print(tabla[tabla['rama'] == rama].drop(columns='rama').to_string(index=False))


=== intermitente ===
       config  wape  wape_piso  mejora_vs_piso  piso_ic_bajo  piso_ic_alto  mejora_vs_v1  v1_ic_bajo  v1_ic_alto  mejora_vs_v1_f2a4  f2a4_ic_bajo  f2a4_ic_alto  mejora_vs_v1_f5  f5_ic_bajo  f5_ic_alto
           v1 0.433      0.438           0.012        -0.061         0.056           NaN         NaN         NaN                NaN           NaN           NaN              NaN         NaN         NaN
   v1+memoria 0.431      0.438           0.016        -0.056         0.059         0.004       0.002       0.006              0.004         0.002         0.008            0.003       0.001       0.005
  v1+momentum 0.431      0.438           0.016        -0.055         0.059         0.004       0.002       0.006              0.004         0.002         0.006            0.004       0.001       0.007
  v1+contexto 0.432      0.438           0.013        -0.061         0.058         0.001      -0.002       0.004              0.001        -0.003         0.004            0.0

## 4. Sensibilidad al producto de mayor pérdida

09 mostró que un producto (P1632) domina el resultado de `intermitente`. Se repite la comparación contra el
piso sin el producto que más exceso de error concentra en cada configuración.

In [5]:
filas = []
for rama, g in res.groupby('rama'):
    for cfg in CONFIGS:
        s = ev.mejora_sin_peor_producto(g, Y, f'pred_{cfg}', PISO, B, SEMILLA)
        filas.append(dict(rama=rama, config=cfg, producto_excluido=s['producto_excluido'],
                          mejora_vs_piso_sin_peor=s['mejora_rel'], ic_bajo=s['ic_bajo'], ic_alto=s['ic_alto']))
sens = pd.DataFrame(filas)
for rama in sens['rama'].unique():
    print(f'\n=== {rama} ===')
    print(sens[sens['rama'] == rama].drop(columns='rama').to_string(index=False))


=== intermitente ===
       config producto_excluido  mejora_vs_piso_sin_peor  ic_bajo  ic_alto
           v1             P1632                    0.049    0.040    0.057
   v1+memoria             P1632                    0.052    0.044    0.060
  v1+momentum             P1632                    0.052    0.043    0.060
  v1+contexto             P1632                    0.050    0.040    0.060
v1+calendario             P1632                    0.050    0.042    0.058
  v2_completa             P1632                    0.051    0.042    0.060

=== suave_no_perecedero ===
       config producto_excluido  mejora_vs_piso_sin_peor  ic_bajo  ic_alto
           v1              P488                    0.017   -0.027    0.054
   v1+memoria             P1631                   -0.013   -0.067    0.024
  v1+momentum             P1460                    0.009   -0.034    0.044
  v1+contexto              P912                   -0.055   -0.141    0.005
v1+calendario              P488                  

## 5. Decisión mecánica

Por rama: se elige la configuración con mayor mejora sobre v1 en los folds de búsqueda (2-4); se **adopta**
solo si (a) el IC95 de esa mejora en 2-4 queda sobre 0 y (b) el fold 5, visto por primera vez, no la contradice
(mejora puntual > 0). Si ninguna cumple, la rama se queda con v1.

In [6]:
decision = {'notebook': '07c_ablacion_features', 'fecha_generacion': pd.Timestamp.now().isoformat(),
            'politica_holdout': ev.POLITICA_HOLDOUT, 'hiperparametros': 'fijos (MODELOS_POR_RAMA, sin busqueda)',
            'replicacion_08': rep.round(4).to_dict('index'), 'ramas': {}}
for rama in tabla['rama'].unique():
    t = tabla[(tabla['rama'] == rama) & (tabla['config'] != 'v1')]
    mejor = t.loc[t['mejora_vs_v1_f2a4'].idxmax()]
    adopta = bool(mejor['f2a4_ic_bajo'] > 0 and mejor['mejora_vs_v1_f5'] > 0)
    decision['ramas'][rama] = dict(
        mejor_config=mejor['config'], mejora_vs_v1_f2a4=float(mejor['mejora_vs_v1_f2a4']),
        ic_f2a4=[float(mejor['f2a4_ic_bajo']), float(mejor['f2a4_ic_alto'])],
        mejora_vs_v1_f5=float(mejor['mejora_vs_v1_f5']), ic_f5=[float(mejor['f5_ic_bajo']), float(mejor['f5_ic_alto'])],
        mejora_vs_piso_config=float(mejor['mejora_vs_piso']),
        mejora_vs_piso_v1=float(tabla[(tabla['rama'] == rama) & (tabla['config'] == 'v1')]['mejora_vs_piso'].iloc[0]),
        adopta=adopta, features_adoptadas=CONFIGS[mejor['config']] if adopta else CONFIGS['v1'])
    print(f'{rama}: mejor config {mejor["config"]} | vs v1 en folds 2-4 {mejor["mejora_vs_v1_f2a4"]:+.1%} '
          f'IC [{mejor["f2a4_ic_bajo"]:+.1%}, {mejor["f2a4_ic_alto"]:+.1%}] | fold 5 {mejor["mejora_vs_v1_f5"]:+.1%} -> '
          + ('ADOPTA' if adopta else 'NO adopta (queda v1)'))
with open(f'{DW}/decision_07c.json', 'w') as f:
    json.dump(decision, f, indent=2, ensure_ascii=False)
tabla.to_csv(f'{DW}/resultados_ablacion_07c.csv', index=False)
sens.to_csv(f'{DW}/sensibilidad_peor_producto_07c.csv', index=False)
res.to_parquet(f'{DW}/preds_ablacion_07c.parquet', index=False)
huella = {'notebook': '07c_ablacion_features', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'entradas': registrar_huella(['matriz_as_of_v2.parquet', 'evaluacion_nivel_1.parquet']),
          'salidas': registrar_huella(['decision_07c.json', 'resultados_ablacion_07c.csv', 'preds_ablacion_07c.parquet'])}
with open(f'{DW}/huella_07c.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
print('guardado: decision_07c.json, resultados_ablacion_07c.csv, sensibilidad_peor_producto_07c.csv, preds_ablacion_07c.parquet, huella_07c.json')

intermitente: mejor config v1+memoria | vs v1 en folds 2-4 +0.4% IC [+0.2%, +0.8%] | fold 5 +0.3% -> ADOPTA
suave_no_perecedero: mejor config v1+momentum | vs v1 en folds 2-4 +0.1% IC [-1.2%, +2.0%] | fold 5 -3.2% -> NO adopta (queda v1)
suave_perecedero: mejor config v1+momentum | vs v1 en folds 2-4 +2.4% IC [-0.2%, +14.3%] | fold 5 -2.3% -> NO adopta (queda v1)


guardado: decision_07c.json, resultados_ablacion_07c.csv, sensibilidad_peor_producto_07c.csv, preds_ablacion_07c.parquet, huella_07c.json


## 6. Registro en MLflow

Un run por (rama, configuración).

In [7]:
for _, r in tabla.iterrows():
    metrics = {'wape': r['wape'], 'wape_piso': r['wape_piso'], 'mejora_vs_piso': r['mejora_vs_piso'],
               'piso_ic_bajo': r['piso_ic_bajo'], 'piso_ic_alto': r['piso_ic_alto']}
    if r['config'] != 'v1':
        metrics.update({k: r[k] for k in ['mejora_vs_v1', 'v1_ic_bajo', 'v1_ic_alto', 'mejora_vs_v1_f2a4', 'f2a4_ic_bajo',
                                          'f2a4_ic_alto', 'mejora_vs_v1_f5', 'f5_ic_bajo', 'f5_ic_alto']})
    s = sens[(sens['rama'] == r['rama']) & (sens['config'] == r['config'])].iloc[0]
    metrics['mejora_vs_piso_sin_peor_producto'] = s['mejora_vs_piso_sin_peor']
    ev.registrar_mlflow(EXPERIMENTO, f"{r['rama']}__{r['config']}",
        tags={'fase': 'B', 'notebook': '07c', 'rama': r['rama'], 'config': r['config'], 'rama_git': 'fix/eda-modelado-nivel1',
              'adopta': str(decision['ramas'][r['rama']]['adopta'] and decision['ramas'][r['rama']]['mejor_config'] == r['config'])},
        params={'features': ','.join(CONFIGS[r['config']]), 'n_features': len(CONFIGS[r['config']]),
                'hiperparametros': 'fijos MODELOS_POR_RAMA', 'bootstrap_B': B, 'huella_matriz_v2': huella['entradas']['matriz_as_of_v2.parquet']},
        metrics=metrics, artefactos=[f'{DW}/decision_07c.json'] if r['config'] == 'v2_completa' else ())
print(f'{len(tabla)} runs registrados en {EXPERIMENTO}')

🏃 View run intermitente__v1 at: http://127.0.0.1:5000/#/experiments/16/runs/a582a12005034b2d9ff1f723588be753
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


🏃 View run intermitente__v1+memoria at: http://127.0.0.1:5000/#/experiments/16/runs/aed2db13e26241b399b7fd455e402471
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


🏃 View run intermitente__v1+momentum at: http://127.0.0.1:5000/#/experiments/16/runs/7c53316d56354b1e93e06db04ed37e0e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


🏃 View run intermitente__v1+contexto at: http://127.0.0.1:5000/#/experiments/16/runs/494c4e2920394463aa058212b44dd033
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


🏃 View run intermitente__v1+calendario at: http://127.0.0.1:5000/#/experiments/16/runs/6671d1521f2f4d5fae494d7c8947b0a4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


...

🏃 View run suave_perecedero__v1+memoria at: http://127.0.0.1:5000/#/experiments/16/runs/6a5aaadd925f4783810fc4f33d8e87fb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


🏃 View run suave_perecedero__v1+momentum at: http://127.0.0.1:5000/#/experiments/16/runs/d2652c7dde574a1985aacb6564e858e2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


🏃 View run suave_perecedero__v1+contexto at: http://127.0.0.1:5000/#/experiments/16/runs/77c9b797f9154b08901a76e48aeee901
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


🏃 View run suave_perecedero__v1+calendario at: http://127.0.0.1:5000/#/experiments/16/runs/0a19e64f8fbe4eedba29eb2476d8dcbe
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16


🏃 View run suave_perecedero__v2_completa at: http://127.0.0.1:5000/#/experiments/16/runs/27dff4c6ee1d4660ab320c2e92255870
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16
18 runs registrados en EDA-fix-nivel1-07c-ablacion-features
